In [1]:
import rlcard
import random

In [2]:
env = rlcard.make('limit-holdem')
state, player_id = env.reset()

In [3]:
step_num = 0

while not env.is_over():
    print(f"\n--- Step {step_num} ---")
    print("Player ID:", player_id)
    print("State:", state)
    print("Legal actions:", state["legal_actions"])

    legal_actions = list(state["legal_actions"].keys())
    action = random.choice(legal_actions)
    print("Chosen action:", action)

    state, player_id = env.step(action)
    step_num += 1

print("\nFinal payoffs:", env.get_payoffs())


--- Step 0 ---
Player ID: 0
State: {'legal_actions': OrderedDict({0: None, 1: None, 2: None}), 'obs': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1.,
       0., 0., 0., 0.]), 'raw_obs': {'hand': ['H7', 'H6'], 'public_cards': [], 'all_chips': [1, 2], 'my_chips': 1, 'legal_actions': ['call', 'raise', 'fold'], 'raise_nums': [0, 0, 0, 0]}, 'raw_legal_actions': ['call', 'raise', 'fold'], 'action_record': []}
Legal actions: OrderedDict({0: None, 1: None, 2: None})
Chosen action: 2

Final payoffs: [-0.5  0.5]


In [4]:
# helper functions to help determine state

# how many cards on the board
def get_street(public_cards):
    if len(public_cards) == 0:
        return 0   # preflop
    elif len(public_cards) == 3:
        return 1   # flop
    elif len(public_cards) == 4:
        return 2   # turn
    else:
        return 3   # river

# determine hand strength
def get_hand_strength_bucket(hand):
    ranks = [card[1] for card in hand]

    rank_map = {
        '2': 2, '3': 3, '4': 4, '5': 5, '6': 6,
        '7': 7, '8': 8, '9': 9, 'T': 10,
        'J': 11, 'Q': 12, 'K': 13, 'A': 14
    }

    values = sorted([rank_map[r] for r in ranks], reverse=True)

    # pair
    if ranks[0] == ranks[1]:
        if values[0] >= 10:
            return 2
        else:
            return 1

    # non-pair
    if values[0] >= 13 and values[1] >= 10:
        return 2
    elif values[0] >= 10:
        return 1
    else:
        return 0

# determine the state
def encode_state(state, player_id):
    raw = state["raw_obs"]

    street = get_street(raw["public_cards"])
    hand_bucket = get_hand_strength_bucket(raw["hand"])
    my_chips = raw["my_chips"]
    opponent_chips = raw["all_chips"][1 - player_id]
    raises_so_far = sum(raw["raise_nums"])

    return (street, hand_bucket, my_chips, opponent_chips, raises_so_far)

In [5]:
# example state

env = rlcard.make('limit-holdem')
state, player_id = env.reset()

encoded = encode_state(state, player_id)
print("Encoded state:", encoded)

Encoded state: (0, 1, 1, 2, 0)


In [6]:
# example transition

current_encoded = encode_state(state, player_id)

# choose a legal action
legal_actions = list(state["legal_actions"].keys())
action = random.choice(legal_actions)

# step forward
next_state, next_player_id = env.step(action)

done = env.is_over()

# encode next state only if game is not over
if done:
    reward = env.get_payoffs()[player_id]
    next_encoded = None
else:
    reward = 0
    next_encoded = encode_state(next_state, next_player_id)

transition = (current_encoded, action, reward, next_encoded, done)
print("Transition:", transition)

Transition: ((0, 1, 1, 2, 0), 1, 0, (0, 2, 2, 4, 1), False)


In [7]:
env = rlcard.make('limit-holdem')
state, player_id = env.reset()

transitions = []

while not env.is_over():
    current_encoded = encode_state(state, player_id)
    legal_actions = list(state["legal_actions"].keys())
    action = random.choice(legal_actions)

    next_state, next_player_id = env.step(action)
    done = env.is_over()

    if done:
        reward = env.get_payoffs()[player_id]
        next_encoded = None
    else:
        reward = 0
        next_encoded = encode_state(next_state, next_player_id)

    transition = (current_encoded, action, reward, next_encoded, done)
    transitions.append(transition)

    state, player_id = next_state, next_player_id

print("Transitions:")
for t in transitions:
    print(t)

print("Final payoffs:", env.get_payoffs())

Transitions:
((0, 0, 1, 2, 0), 2, np.float64(-0.5), None, True)
Final payoffs: [-0.5  0.5]


In [8]:
# complete run through example

env = rlcard.make('limit-holdem')
all_transitions = []

num_hands = 10

for _ in range(num_hands):
    state, player_id = env.reset()

    while not env.is_over():
        current_encoded = encode_state(state, player_id)
        legal_actions = list(state["legal_actions"].keys())
        action = random.choice(legal_actions)

        next_state, next_player_id = env.step(action)
        done = env.is_over()

        if done:
            reward = float(env.get_payoffs()[player_id])
            next_encoded = None
        else:
            reward = 0
            next_encoded = encode_state(next_state, next_player_id)

        transition = {
            "state": current_encoded,
            "action": action,
            "reward": reward,
            "next_state": next_encoded,
            "done": done
        }
        all_transitions.append(transition)

        state, player_id = next_state, next_player_id

print("Total transitions collected:", len(all_transitions))
print("First 5 transitions:")
for t in all_transitions[:5]:
    print(t)

Total transitions collected: 24
First 5 transitions:
{'state': (0, 1, 1, 2, 0), 'action': 2, 'reward': -0.5, 'next_state': None, 'done': True}
{'state': (0, 1, 1, 2, 0), 'action': 1, 'reward': 0, 'next_state': (0, 0, 2, 4, 1), 'done': False}
{'state': (0, 0, 2, 4, 1), 'action': 1, 'reward': 0, 'next_state': (0, 1, 4, 6, 2), 'done': False}
{'state': (0, 1, 4, 6, 2), 'action': 1, 'reward': 0, 'next_state': (0, 0, 6, 8, 3), 'done': False}
{'state': (0, 0, 6, 8, 3), 'action': 2, 'reward': -3.0, 'next_state': None, 'done': True}
